# Florida top-city population forecast walkthrough

This notebook is a guided run-through of `florida_top5_forecast.py`. It
reuses the exact same modeling code as `charlotte_walkthrough.ipynb` (in the
`charlotte/` folder) -- damped-trend ETS and ARIMA(1,1,1), picked by rolling
backtest -- applied to Florida's largest cities instead of a single one.

The population history here comes from the Florida Office of Economic &
Demographic Research's municipal estimates (`FLmupops.xlsx`, in this same
folder), not the Census API -- that file ships with 40+ years of annual data
for every incorporated Florida city, so everything in this notebook runs
fully offline, no API key needed. See `edr_population.py`'s docstring for
why this data source was chosen over ACS1.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from edr_population import build_panel, city_series, list_cities, filter_florida, clean_city_name
from charlotte_5y import HORIZON
from florida_top5_forecast import run_city, TARGET_CITIES, MIN_OBS

%matplotlib inline

## 1. Load the municipal population panel

`build_panel` reads every yearly sheet of the EDR workbook and returns one
long table: (year, place_id, city, county, population). `filter_florida` is a
no-op here (the source is already Florida-only) but is kept so this script's
interface matches `census_devs.py`'s, in case you want to swap sources.

In [ ]:
panel = build_panel()
fl_panel = filter_florida(panel)
print(f"{fl_panel['city'].nunique()} Florida municipalities, "
      f"{fl_panel['year'].min()}-{fl_panel['year'].max()}")

## 2. Pick out the target cities

This script targets a fixed short list (see `TARGET_CITIES`); change that
tuple in `florida_top5_forecast.py` to forecast a different set of Florida
cities -- `list_cities` returns every city available if you want to browse
the full list first.

In [ ]:
print("Target cities:", TARGET_CITIES)

all_cities = list_cities(fl_panel)
targets = all_cities[all_cities["city"].isin(TARGET_CITIES)]
targets.assign(population=targets["population"].map("{:,.0f}".format))

## 3. Backtest + forecast each city

`run_city` is the per-city version of the same backtest-then-forecast
pipeline used in `charlotte_walkthrough.ipynb`: fit ETS and ARIMA, rolling
5-year-ahead backtest to see which one actually predicted better, then
forecast forward with the winner's confidence interval.

In [ ]:
alpha = 0.05  # 95% confidence interval
results = []

for place_id, meta in targets.iterrows():
    yrs, pop = city_series(fl_panel, place_id)
    if len(pop) < MIN_OBS:
        print(f"{meta['city']}: skipped, only {len(pop)} years of data")
        continue
    label = clean_city_name(meta["city"])
    results.append(run_city(label, yrs, pop, alpha=alpha, show=True))

plt.show()

## 4. Summary table

Same shape as `florida_top5_forecast.csv` in this folder -- this cell
regenerates it from scratch rather than reading the saved copy.

In [ ]:
import pandas as pd

pd.DataFrame(results)

## Adapting this to other cities

Nothing in `run_city` or the modeling functions it calls is Florida-specific
-- swap `TARGET_CITIES` for any municipality names present in the EDR
workbook, or swap the data source entirely (`../population_matrix/census_devs.py`
exposes the same `build_panel`/`city_series` interface against the Census
ACS1 API for any US city, not just Florida).